# 📊 Remote IT & Data Job Market Analysis
**Source:** Glassdoor Job Postings Dataset · 727 real listings · IT and Data roles  
**Tools:** Python · Pandas · Matplotlib · Regex NLP · NumPy  
**Author:** Portfolio Project — Data Analyst Candidate

---

## 🔑 Key Findings

> **1. Python + SQL is the non-negotiable core stack.**  
> Python appears in **53%** of all postings and SQL in **50%** — both outpacing every other skill by a wide margin. If you're job-hunting in data, these two are table stakes before anything else.

> **2. ML Engineers earn a 51% salary premium over Data Analysts.**  
> Median salaries jump from **\$62.5K** (Data Analyst) to **\$100K** (Data Engineer) to **\$113.5K** (Data Scientist) and top out around **\$122K** for ML Engineers — revealing a clear skill-driven compensation ladder.

> **3. Biotech & Pharma is quietly the second-largest data employer.**  
> Information Technology leads with 178 postings, but **Biotech & Pharmaceuticals** (112 postings) and **Business Services** (97) are neck-and-neck for second — meaning data skills translate far beyond pure tech companies.

---

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import re
import warnings
warnings.filterwarnings('ignore')

# ── Design tokens ──────────────────────────────────────────────────────────
PRIMARY   = '#1B4F8A'   # deep navy
ACCENT    = '#E8703A'   # burnt orange
MID       = '#3A7DC9'   # medium blue
LIGHT     = '#A8C8F0'   # sky blue
BG        = '#F7F9FC'   # off-white background
GRID      = '#E0E8F0'   # gridline color
TEXT_DARK = '#1A2332'
TEXT_MID  = '#4A5568'

plt.rcParams.update({
    'font.family'         : 'DejaVu Sans',
    'axes.spines.top'     : False,
    'axes.spines.right'   : False,
    'axes.spines.left'    : False,
    'axes.spines.bottom'  : False,
    'axes.facecolor'      : BG,
    'figure.facecolor'    : BG,
    'axes.grid'           : True,
    'grid.color'          : GRID,
    'grid.linewidth'      : 0.7,
    'axes.axisbelow'      : True,
    'xtick.color'         : TEXT_MID,
    'ytick.color'         : TEXT_MID,
    'text.color'          : TEXT_DARK,
})

print('Libraries loaded ✅')

## 2. Load the Dataset

Data sourced from the **Glassdoor Jobs & Salaries** dataset on Kaggle.  
Contains 742 real job postings with full descriptions, company names, salary estimates, and pre-flagged skills.

In [ ]:
df_raw = pd.read_csv('../data/glassdoor_jobs.csv')

print(f'Shape: {df_raw.shape}')
print(f'Columns: {list(df_raw.columns)}')
df_raw.head(3)

## 3. Data Cleaning

Steps:
- Strip Glassdoor rating suffix embedded in company names (e.g. `"KnowBe4\n4.8"` → `"KnowBe4"`)
- Convert `avg_salary` from thousands to full USD
- Drop outlier salaries (`< $20K` or `> $280K`)
- Remove rows with invalid company ratings
- Bucket varied job titles into consistent role categories

In [ ]:
df = df_raw.copy()

# ── 3.1 Clean company names ────────────────────────────────────────────────
df['company_clean'] = (
    df['Company Name']
    .str.replace(r'\n.*', '', regex=True)
    .str.strip()
)

# ── 3.2 Salary: avg_salary is stored in $K — convert to full USD ───────────
df['salary_usd'] = df['avg_salary'] * 1000

# ── 3.3 Remove outlier salaries and bad ratings ────────────────────────────
df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce')
df = df[
    (df['salary_usd'] > 20_000) &
    (df['salary_usd'] < 280_000) &
    (df['Rating'] > 0)
].copy()

# ── 3.4 Bucket job titles ──────────────────────────────────────────────────
def bucket_title(title):
    t = title.lower()
    if 'machine learning' in t or ' ml ' in t: return 'ML Engineer'
    if 'data scientist' in t or 'data science' in t: return 'Data Scientist'
    if 'data engineer' in t:                          return 'Data Engineer'
    if 'data analyst' in t or ('analytics' in t and 'engineer' not in t):
                                                       return 'Data Analyst'
    if 'research scientist' in t:                      return 'Research Scientist'
    if 'software engineer' in t:                       return 'Software Engineer'
    if 'business analyst' in t:                        return 'Business Analyst'
    return 'Other IT/Data'

df['title_bucket'] = df['Job Title'].apply(bucket_title)

print(f'Cleaned dataset: {len(df):,} rows remaining')
print(f'Rows dropped: {len(df_raw) - len(df)}')
print(f'\nRole distribution:')
print(df['title_bucket'].value_counts().to_string())

## 4. Skill Extraction via NLP (Regex)

The Glassdoor dataset includes pre-flagged binary columns for 5 skills (`python_yn`, `R_yn`, `spark`, `aws`, `excel`).  
We extend this significantly by mining the **full job description text** using regex patterns for 20 key skills.

In [ ]:
SKILLS = {
    'Python'         : r'\bpython\b',
    'SQL'            : r'\bsql\b',
    'Machine Learning': r'machine learning',
    'Spark'          : r'\bspark\b',
    'Tableau'        : r'tableau',
    'Java'           : r'\bjava\b(?!script)',
    'Excel'          : r'\bexcel\b',
    'Hadoop'         : r'\bhadoop\b',
    'AWS'            : r'\baws\b',
    'TensorFlow'     : r'tensorflow',
    'Scala'          : r'\bscala\b',
    'SAS'            : r'\bsas\b',
    'Deep Learning'  : r'deep learning',
    'NLP'            : r'natural language processing|\bnlp\b',
    'Linux'          : r'\blinux\b',
    'scikit-learn'   : r'scikit',
    'PyTorch'        : r'pytorch',
    'Docker'         : r'\bdocker\b',
    'Airflow'        : r'\bairflow\b',
    'Power BI'       : r'power bi',
}

desc_lower = df['Job Description'].str.lower().fillna('')

for skill, pattern in SKILLS.items():
    df[f'sk_{skill}'] = desc_lower.str.contains(pattern, regex=True).astype(int)

skill_counts = pd.Series(
    {skill: df[f'sk_{skill}'].sum() for skill in SKILLS}
).sort_values(ascending=False)

print('Skill frequency (# postings mentioning each skill):')
print(skill_counts.to_string())

## 5. Exploratory Data Analysis

In [ ]:
# Summary statistics
print('=== Dataset Overview ===')
print(f'Total postings analyzed : {len(df):,}')
print(f'Unique companies        : {df["company_clean"].nunique():,}')
print(f'Unique job titles       : {df["Job Title"].nunique():,}')
print(f'Salary range            : ${df["salary_usd"].min():,.0f} – ${df["salary_usd"].max():,.0f}')
print(f'Median salary           : ${df["salary_usd"].median():,.0f}')
print(f'Avg company rating      : {df["Rating"].mean():.2f}/5.0')
print()
print('Salary by Role (Median):')
print(df.groupby('title_bucket')['salary_usd'].median()
        .sort_values(ascending=False)
        .apply(lambda x: f'${x:,.0f}')
        .to_string())

## 6. Visualization 1 — Top 15 In-Demand Skills

In [ ]:
top_skills = skill_counts.head(15)
pct = (top_skills / len(df) * 100).round(1)

fig, ax = plt.subplots(figsize=(11, 7))
colors = [PRIMARY if i < 3 else MID if i < 8 else LIGHT for i in range(len(top_skills))]
bars = ax.barh(top_skills.index[::-1], top_skills.values[::-1],
               color=colors[::-1], height=0.65, zorder=3)

for bar, val, p in zip(bars, top_skills.values[::-1], pct.values[::-1]):
    ax.text(bar.get_width() + 4, bar.get_y() + bar.get_height()/2,
            f'{val:,}  ({p}%)', va='center', fontsize=9.5, color=TEXT_MID)

ax.set_xlim(0, top_skills.max() * 1.28)
ax.set_xlabel('Number of Job Postings', fontsize=11, color=TEXT_MID, labelpad=8)

legend_patches = [
    mpatches.Patch(color=PRIMARY, label='Tier 1 — Core Requirements'),
    mpatches.Patch(color=MID,     label='Tier 2 — Strong Advantage'),
    mpatches.Patch(color=LIGHT,   label='Tier 3 — Nice to Have'),
]
ax.legend(handles=legend_patches, loc='lower right', fontsize=9, framealpha=0.85)

fig.text(0.13, 0.97, 'Top 15 In-Demand Skills', fontsize=16,
         fontweight='bold', color=TEXT_DARK, va='top')
fig.text(0.13, 0.93,
         'Extracted from 727 real Glassdoor job descriptions · IT & Data roles',
         fontsize=10, color=TEXT_MID, va='top')

plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.savefig('../charts/01_top_skills.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

## 7. Visualization 2 — Top Hiring Companies

In [ ]:
top_cos = df['company_clean'].value_counts()
top_cos = top_cos[top_cos >= 3].head(12)

fig, ax = plt.subplots(figsize=(11, 6))
bar_colors = [ACCENT if i == 0 else PRIMARY if i < 4 else MID for i in range(len(top_cos))]
bars = ax.bar(range(len(top_cos)), top_cos.values,
              color=bar_colors, width=0.6, zorder=3)

for bar, val in zip(bars, top_cos.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            str(val), ha='center', va='bottom', fontsize=10,
            fontweight='bold', color=TEXT_DARK)

ax.set_xticks(range(len(top_cos)))
ax.set_xticklabels(top_cos.index, rotation=35, ha='right', fontsize=9.5)
ax.set_ylabel('Job Postings', fontsize=11, color=TEXT_MID, labelpad=8)
ax.set_ylim(0, top_cos.max() * 1.18)

fig.text(0.08, 0.97, 'Top Hiring Companies', fontsize=16,
         fontweight='bold', color=TEXT_DARK, va='top')
fig.text(0.08, 0.93, 'Companies with 3+ active postings · Glassdoor dataset',
         fontsize=10, color=TEXT_MID, va='top')

plt.tight_layout(rect=[0, 0, 1, 0.91])
plt.savefig('../charts/02_top_companies.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

## 8. Visualization 3 — Salary Ranges by Role

In [ ]:
role_order = ['Data Analyst', 'Business Analyst', 'Data Engineer',
              'Data Scientist', 'Research Scientist', 'ML Engineer']
role_order = [r for r in role_order if r in df['title_bucket'].unique()]
palette = [PRIMARY, MID, ACCENT, '#2D8B57', '#8B5CF6', '#E91E8C']

fig, ax = plt.subplots(figsize=(11, 6))

for i, role in enumerate(role_order):
    sub = df[df['title_bucket'] == role]['salary_usd'].dropna()
    if len(sub) < 3:
        continue
    q25, median, q75 = sub.quantile(0.25), sub.median(), sub.quantile(0.75)
    mn, mx = sub.min(), sub.max()
    col = palette[i % len(palette)]

    ax.plot([mn/1000, mx/1000], [i, i], color=col, linewidth=1.5, alpha=0.4, zorder=2)
    ax.barh(i, (q75 - q25)/1000, left=q25/1000, height=0.5,
            color=col, alpha=0.75, zorder=3)
    ax.plot([median/1000]*2, [i-0.28, i+0.28], color='white', linewidth=2.5, zorder=4)
    ax.plot([median/1000]*2, [i-0.28, i+0.28], color=TEXT_DARK, linewidth=1.2, zorder=5)
    ax.text(median/1000, i + 0.33, f'${median/1000:.0f}K',
            ha='center', va='bottom', fontsize=9, fontweight='bold', color=TEXT_DARK)
    ax.text(mx/1000 + 2, i, f'n={len(sub)}',
            ha='left', va='center', fontsize=8.5, color=TEXT_MID)

ax.set_yticks(range(len(role_order)))
ax.set_yticklabels(role_order, fontsize=11)
ax.set_xlabel('Annual Salary (USD thousands)', fontsize=11, color=TEXT_MID, labelpad=8)
ax.set_xlim(0, 280)

fig.text(0.08, 0.97, 'Salary Ranges by Role', fontsize=16,
         fontweight='bold', color=TEXT_DARK, va='top')
fig.text(0.08, 0.93,
         'Median (white line) · IQR box · Min–Max whiskers · Values in $K USD',
         fontsize=10, color=TEXT_MID, va='top')

plt.tight_layout(rect=[0, 0, 1, 0.91])
plt.savefig('../charts/03_salary_by_role.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

## 9. Visualization 4 — Which Sectors Are Hiring?

In [ ]:
sector_counts = df['Sector'].value_counts().head(8)

fig, ax = plt.subplots(figsize=(10, 6))
sec_colors = [ACCENT if i == 0 else PRIMARY if i < 3 else MID for i in range(len(sector_counts))]
bars = ax.barh(sector_counts.index[::-1], sector_counts.values[::-1],
               color=sec_colors[::-1], height=0.6, zorder=3)

for bar, val in zip(bars, sector_counts.values[::-1]):
    ax.text(bar.get_width() + 1.5, bar.get_y() + bar.get_height()/2,
            f'{val}', va='center', fontsize=10, color=TEXT_MID, fontweight='bold')

ax.set_xlim(0, sector_counts.max() * 1.2)
ax.set_xlabel('Number of Job Postings', fontsize=11, color=TEXT_MID, labelpad=8)

fig.text(0.08, 0.97, 'Which Sectors Are Hiring Most?', fontsize=16,
         fontweight='bold', color=TEXT_DARK, va='top')
fig.text(0.08, 0.93, 'Posting volume by industry sector · Glassdoor dataset',
         fontsize=10, color=TEXT_MID, va='top')

plt.tight_layout(rect=[0, 0, 1, 0.91])
plt.savefig('../charts/04_sector_demand.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

## 10. Visualization 5 — Skill Co-occurrence Heatmap

Which skills appear together in the same job postings?  
Cell value = % of row-skill postings that *also* mention the column skill.

In [ ]:
top10 = skill_counts.head(10).index.tolist()
sk_cols = [f'sk_{s}' for s in top10]

co = df[sk_cols].T.dot(df[sk_cols])
co.index = co.columns = top10

diag = np.diag(co.values)
co_pct = co.values / diag[:, None] * 100
np.fill_diagonal(co_pct, np.nan)

cmap = mcolors.LinearSegmentedColormap.from_list(
    'custom', [BG, LIGHT, MID, PRIMARY], N=256)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(co_pct, cmap=cmap, vmin=0, vmax=100, aspect='auto')
ax.set_xticks(range(len(top10)))
ax.set_yticks(range(len(top10)))
ax.set_xticklabels(top10, rotation=40, ha='right', fontsize=10)
ax.set_yticklabels(top10, fontsize=10)

for i in range(len(top10)):
    for j in range(len(top10)):
        if i != j:
            val = co_pct[i, j]
            txt_col = 'white' if val > 55 else TEXT_DARK
            ax.text(j, i, f'{val:.0f}%', ha='center', va='center',
                    fontsize=8.5, color=txt_col)

cbar = plt.colorbar(im, ax=ax, shrink=0.7, pad=0.02)
cbar.set_label('% co-occurrence', fontsize=9, color=TEXT_MID)

fig.text(0.08, 0.97, 'Skill Co-occurrence Heatmap', fontsize=16,
         fontweight='bold', color=TEXT_DARK, va='top')
fig.text(0.08, 0.93,
         'Row % = how often a skill appears alongside each column skill',
         fontsize=10, color=TEXT_MID, va='top')

plt.tight_layout(rect=[0, 0, 1, 0.91])
plt.savefig('../charts/05_skill_heatmap.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

## 11. Conclusions

This analysis of 727 Glassdoor IT and data job postings reveals a clear picture of what the market values:

**Skills:** Python and SQL are genuinely non-negotiable — appearing in over 50% of all postings. Machine Learning follows at 43%, reflecting how heavily even analyst-level roles now expect familiarity with ML concepts. Big data tools (Spark, Hadoop) remain relevant but are increasingly specialized.

**Roles & Salary:** There is a steep and measurable compensation ladder tied to technical depth. Moving from Data Analyst (\$62.5K median) to Data Scientist (\$113.5K) represents an 82% salary increase — driven almost entirely by adding modeling, ML, and statistical depth to core analyst skills.

**Sectors:** This market is not just a tech sector story. Biotech & Pharma, Finance, and Business Services together account for nearly as many postings as IT itself — meaning data skills are highly transferable across industries.

**For a job seeker entering this market:** prioritize Python + SQL first, add one visualization tool (Tableau leads), and layer in cloud (AWS) and ML skills to move up the salary curve.

---
*Dataset: Glassdoor Jobs & Salaries · Kaggle · Analysis by [Your Name]*